In [4]:
import os
os.listdir("/content/drive/MyDrive/data")

['categories - Copy.csv',
 'INvideos - Copy.csv',
 'USvideos - Copy.csv',
 'GBvideos - Copy.csv']

# 1.Project Configuration

In [21]:
import os
from datetime import datetime

# Dataset folder
DATA_FOLDER = "/content/drive/MyDrive/data"

# Countries
COUNTRIES = ["US", "GB", "IN"]

# Category file
CATEGORY_FILE = os.path.join(DATA_FOLDER, "categories - Copy.csv")

# Classification thresholds
THRESHOLDS = {
    "viral_views": 1000000,
    "steady_views": 100000,
    "viral_like_ratio": 0.90,
    "divisive_ratio": 0.5
}

# Run start time
run_time = datetime.now()

print("=" * 45)
print("       TRENDPULSE PROJECT")
print("=" * 45)
print(f"Countries: {', '.join(COUNTRIES)}")
print(f"Run Started: {run_time}")
print(f"Data Folder: {DATA_FOLDER}")
print("=" * 45)

       TRENDPULSE PROJECT
Countries: US, GB, IN
Run Started: 2026-11-13 14:33:27.854670
Data Folder: /content/drive/MyDrive/data


# 2.Create the Loading Function

In [4]:
import pandas as pd
#Required columns for our project
required_columns = [
    "video_id",
    "trending_date",
    "title",
    "channel_title",
    "category_id",
    "publish_time",
    "views",
    "likes",
    "dislikes",
    "comment_count",
    "comments_disabled",
    "ratings_disabled"
]


# Reusable function to load country data
def load_country(path, country, max_rows=None):

    # Read CSV file
    df = pd.read_csv(path, nrows=max_rows)

    # Check required columns
    missing_columns = []

    for column in required_columns:
        if column not in df.columns:
            missing_columns.append(column)

    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}")

    # Convert DataFrame into list of dictionaries
    records = df.to_dict("records")

    # Add country to every record
    for record in records:
        record["country"] = country

    # Display loaded record count
    print(f"{country}: {len(records)} records loaded")

    # Return records
    return records

us_records = load_country(
    os.path.join(DATA_FOLDER, "USvideos - Copy.csv"), "US"
)

gb_records = load_country(
    os.path.join(DATA_FOLDER, "GBvideos - Copy.csv"), "GB"
)

in_records = load_country(
    os.path.join(DATA_FOLDER, "INvideos - Copy.csv"), "IN"
)

US: 16080 records loaded
GB: 15075 records loaded
IN: 15075 records loaded


2.1 Load categories & Create Dictionary

In [5]:
# Load categories CSV
categories_df = pd.read_csv(CATEGORY_FILE)

# Create category dictionary
category_names = {}

for record in categories_df.to_dict("records"):
    category_names[str(record["category_id"])] = record["category_name"]

# Display dictionary
print(category_names)

{'1': 'Film & Animation', '2': 'Autos & Vehicles', '10': 'Music', '15': 'Pets & Animals', '17': 'Sports', '19': 'Travel & Events', '20': 'Gaming', '22': 'People & Blogs', '23': 'Comedy', '24': 'Entertainment', '25': 'News & Politics', '26': 'Howto & Style', '27': 'Education', '28': 'Science & Technology', '29': 'Nonprofits & Activism', '30': 'Movies', '43': 'Shows'}


2.2 Inspect one video record

In [30]:
print("US Dataset Columns:")
print(us_records[0].keys())

print("\nFirst Video Record:")
print(us_records[0])

US Dataset Columns:
dict_keys(['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_time', 'tags', 'views', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description', 'country', 'label'])

First Video Record:
{'video_id': 'g_N-w8LeAi8', 'trending_date': datetime.date(2017, 11, 14), 'title': 'Honest Football Prank LEAKED', 'channel_title': 'QuietSpeech10', 'category_id': 22, 'publish_time': datetime.datetime(2017, 11, 14, 7, 4, 28, tzinfo=datetime.timezone.utc), 'tags': 'Monsoon|Robots', 'views': 48555, 'likes': 2189, 'dislikes': 180, 'comment_count': 136, 'thumbnail_link': 'https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg', 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'description': 'Watch this video. Subscribe for more content about chess. Business enquiries in the channel about page.', 'country': 'US', 'label': 'Low Engagement'}


2.3 Combine all country recoeds

In [7]:
# Combine all country records
all_records = us_records + gb_records + in_records

# Display total records
print("Total Records:", len(all_records))

# Display country-wise records
print("US Records:", len(us_records))
print("GB Records:", len(gb_records))
print("IN Records:", len(in_records))

Total Records: 46230
US Records: 16080
GB Records: 15075
IN Records: 15075


# 3.Type Conversion & Data Cleaning

3.1 Decide Cleaning rules

Data Cleaning Rules
1.Numeric fields (views, likes, dislikes, comment_count): Convert into integers. Missing or invalid values will be replaced with 0.
2.Boolean fields (comments_disabled, ratings_disabled, video_error_or_removed): Convert into True or False. Missing or invalid values will be replaced with False.
3.trending_date: Convert YY.DD.MM format into proper date format. Invalid dates will be replaced with None.
4.publish_time: Convert into datetime format. Invalid dates will be replaced with None.
5.Error handling: Use try/except to prevent invalid values from stopping the program.
6.Record handling: We will retain records instead of deleting them and report conversion failures separately.
Reason: Retaining records helps preserve the original dataset size. Default values allow processing to continue, while failure counts make data quality issues visible.

3.2 Type conversion and cleaning

In [8]:
from datetime import datetime
numeric_fields = ["views", "likes", "dislikes", "comment_count"]
boolean_fields = [
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed"
]
date_fields = ["trending_date", "publish_time"]
parse_failures = {}
for field in numeric_fields + boolean_fields + date_fields:
    parse_failures[field] = 0
def safe_integer(value, field):
    try:
        if value is None or str(value).strip() == "":
            raise ValueError("Blank numeric value")
        return int(float(value))
    except (ValueError, TypeError, OverflowError):
        parse_failures[field] += 1
        return 0
def safe_boolean(value, field):
    try:
        if isinstance(value, bool):
            return value
        text_value = str(value).strip().lower()
        if text_value in ["true", "1", "yes"]:
            return True
        elif text_value in ["false", "0", "no"]:
            return False
        else:
            raise ValueError("Invalid Boolean value")
    except (ValueError, TypeError):
        parse_failures[field] += 1
        return False
def safe_trending_date(value):
    try:
        return datetime.strptime(str(value), "%y.%d.%m").date()
    except (ValueError, TypeError):
        parse_failures["trending_date"] += 1
        return None
def safe_publish_time(value):
    try:
        text_value = str(value).strip()
        return datetime.fromisoformat(
            text_value.replace("Z", "+00:00")
        )
    except (ValueError, TypeError):
        parse_failures["publish_time"] += 1
        return None


cleaned_records = []

for record in all_records:
    for field in numeric_fields:
        record[field] = safe_integer(record.get(field), field)
    for field in boolean_fields:
        record[field] = safe_boolean(record.get(field), field)
    record["trending_date"] = safe_trending_date(
        record.get("trending_date")
    )
    record["publish_time"] = safe_publish_time(
        record.get("publish_time")
    )
    cleaned_records.append(record)
print("Cleaned records:", len(cleaned_records))
print("Parse failures by field:")
for field, count in parse_failures.items():
    print(f"{field}: {count}")


Cleaned records: 46230
Parse failures by field:
views: 0
likes: 12
dislikes: 0
comment_count: 36
comments_disabled: 0
ratings_disabled: 0
video_error_or_removed: 0
trending_date: 0
publish_time: 0


# 4.Core data structures


 4.1List of dictionaries

In [9]:
print("Data type:", type(cleaned_records))
print("First record title:", cleaned_records[0]["title"])
print("Number of records:", len(cleaned_records))

Data type: <class 'list'>
First record title: Honest Football Prank LEAKED
Number of records: 46230


 4.2 Tuples — selected video details

In [29]:
video_tuples = []
for record in cleaned_records:
    one_video = (
        record["video_id"],
        record["trending_date"],
        record["views"],
        record["likes"]
    )
    video_tuples.append(one_video)
print("Example tuple:", video_tuples[0])
print("Tuple count:", len(video_tuples))

Example tuple: ('g_N-w8LeAi8', datetime.date(2017, 11, 14), 48555, 2189)
Tuple count: 46230


 4.3 Sets — unique IDs, intersection and difference

In [11]:
us_ids = set()
gb_ids = set()
in_ids = set()
for record in cleaned_records:
    if record["country"] == "US":
        us_ids.add(record["video_id"])
    elif record["country"] == "GB":
        gb_ids.add(record["video_id"])
    elif record["country"] == "IN":
        in_ids.add(record["video_id"])
us_gb_common = us_ids.intersection(gb_ids)
us_in_common = us_ids.intersection(in_ids)
gb_in_common = gb_ids.intersection(in_ids)
us_only = us_ids.difference(gb_ids, in_ids)
gb_only = gb_ids.difference(us_ids, in_ids)
in_only = in_ids.difference(us_ids, gb_ids)
print("Unique US IDs:", len(us_ids))
print("Unique GB IDs:", len(gb_ids))
print("Unique IN IDs:", len(in_ids))
print("US and GB common IDs:", len(us_gb_common))
print("US and IN common IDs:", len(us_in_common))
print("GB and IN common IDs:", len(gb_in_common))
print("US-only IDs:", len(us_only))
print("GB-only IDs:", len(gb_only))
print("IN-only IDs:", len(in_only))

Unique US IDs: 5068
Unique GB IDs: 4113
Unique IN IDs: 5878
US and GB common IDs: 134
US and IN common IDs: 120
GB and IN common IDs: 126
US-only IDs: 4834
GB-only IDs: 3873
IN-only IDs: 5652


4.4 Dictionary counter + slicing

In [12]:
channel_counts = {}
for record in cleaned_records:
    channel = record["channel_title"]
    if channel not in channel_counts:
        channel_counts[channel] = 0
    channel_counts[channel] += 1
sorted_channels = sorted(
    channel_counts.items(),
    key=lambda item: item[1],
    reverse=True
)
top_15_channels = sorted_channels[:15]
print("Top 15 channels:")
for channel, appearances in top_15_channels:
    print(f"{channel}: {appearances}")

Top 15 channels:
QuietChallenge75: 2464
OfficialReview41: 1586
UltimateUnboxing65: 1433
InsaneReaction46: 1233
Brand NewChallenge45: 1175
CrazyTrailer91: 1018
FinalTrailer17: 911
UltimateSpeech20: 770
Brand NewSecret51: 684
WildHighlights7: 682
SurprisingHighlights41: 650
FinalBreakdown11: 643
CrazyRecipe67: 603
QuietInterview9: 570
UltimateVlog96: 565


 4.5 Nested dictionary — country fi category fi count

In [13]:
country_category_summary = {}
for record in cleaned_records:
    country = record["country"]
    category_id = str(record["category_id"])
    category_name = category_names.get(category_id, "Unknown")
    if country not in country_category_summary:
        country_category_summary[country] = {}
    if category_name not in country_category_summary[country]:
        country_category_summary[country][category_name] = 0
    country_category_summary[country][category_name] += 1
for country, category_counts in country_category_summary.items():
    print(f"\\nCountry: {country}")
    shown = 0
    for category, count in category_counts.items():
        print(f"{category}: {count}")
        shown += 1
        if shown == 5:
            break

\nCountry: US
People & Blogs: 1423
News & Politics: 755
Music: 3052
Education: 561
Entertainment: 4371
\nCountry: GB
Science & Technology: 470
Howto & Style: 588
Comedy: 941
Entertainment: 2844
Music: 5013
\nCountry: IN
Music: 2298
Film & Animation: 770
Entertainment: 5117
Comedy: 966
People & Blogs: 1098


# 5.Variable scope — global and local

In [14]:
project_name = "TrendPulse"
def show_scope():
    local_message = "I am local to this function"
    print("Inside function - global:", project_name)
    print("Inside function - local:", local_message)
    return local_message
returned_message = show_scope()
print("Outside function - global:", project_name)
print("Outside function - returned local value:", returned_message)
try:
    print(local_message)
except NameError:
    print("local_message is not directly available outside the function")

Inside function - global: TrendPulse
Inside function - local: I am local to this function
Outside function - global: TrendPulse
Outside function - returned local value: I am local to this function
local_message is not directly available outside the function


# 6.Loops, range(), break and continue

6.1 title_features(title)

In [15]:
def title_features(title):
    if title is None:
        title = ""
    title = str(title)
    punctuation_count = 0
    for index in range(len(title)):
        if title[index] == "!" or title[index] == "?":
            punctuation_count += 1
    words = title.split()
    uppercase_words = 0
    for word in words:
        if word.isupper() and any(char.isalpha() for char in word):
            uppercase_words += 1
    return {
        "uppercase_words": uppercase_words,
        "exclamation_question_marks": punctuation_count,
        "word_count": len(words)
    }
print(title_features("WOW! Amazing FOOTBALL?"))

{'uppercase_words': 2, 'exclamation_question_marks': 2, 'word_count': 3}


6.2 while + continue + break

In [16]:
view_threshold = THRESHOLDS["steady_views"]
index = 0
found_video = False
while index < len(cleaned_records):
    current_record = cleaned_records[index]
    index += 1
    if current_record["views"] <= view_threshold:
        continue
    print(
        "First video above threshold:",
        current_record["title"],
        "| Views:",
        current_record["views"]
    )
    found_video = True
    break
if not found_video:
    print("No video found above the threshold.")

First video above threshold: Surprising Cricket Review | Views: 198385


# 7.Rule-based classification

Labels: Viral Hit, Steady Performer, Divisive, Low Engagement, Unrated.

In [22]:

def classify_video(record):
    views = record.get("views", 0)
    likes = record.get("likes", 0)
    dislikes = record.get("dislikes", 0)
    ratings_disabled = record.get("ratings_disabled", False)
    comments_disabled = record.get("comments_disabled", False)
    total_ratings = likes + dislikes
    # Prevent division by zero and handle unavailable ratings
    if ratings_disabled or total_ratings == 0:
        return "Unrated"
    like_ratio = likes / total_ratings
    if views == 0:
        return "Low Engagement"
    elif views >= THRESHOLDS["viral_views"]:
        if like_ratio >= THRESHOLDS["viral_like_ratio"]:
            if comments_disabled:
                return "Steady Performer"
            else:
                return "Viral Hit"
        else:
            return "Divisive"
    elif views >= THRESHOLDS["steady_views"]:
        if like_ratio < THRESHOLDS["divisive_ratio"]:
            return "Divisive"
        else:
            return "Steady Performer"
    else:
        if like_ratio < THRESHOLDS["divisive_ratio"]:
            return "Divisive"
        else:
            return "Low Engagement"
label_counts = {
    "Viral Hit": 0,
    "Steady Performer": 0,
    "Divisive": 0,
    "Low Engagement": 0,
    "Unrated": 0
}
for record in cleaned_records:
    label = classify_video(record)
    record["label"] = label
    label_counts[label] += 1
print("Classification counts:")
for label, count in label_counts.items():
    print(f"{label}: {count}")
print("Total classified:", sum(label_counts.values()))

Classification counts:
Viral Hit: 8654
Steady Performer: 27535
Divisive: 775
Low Engagement: 8722
Unrated: 544
Total classified: 46230


7.1 Hand-trace

Example record:

views = 1,500,000

likes = 900

dislikes = 100

ratings_disabled = False

comments_disabled = False

Step 1: Ratings enabled; total_ratings = 900 + 100 = 1,000.

Step 2: like_ratio = 900 / 1,000 = 0.90.

Step 3: Views are not zero and are at least 1,000,000.

Step 4: like_ratio 0.90 is at least 0.70.
Step 5: comments are enabled.

Result: Viral Hit.

Edge cases: ratings_disabled=True fi Unrated. likes + dislikes = 0 fi Unrated (prevents division by zero). views = 0 fi
Low Engagement, provided ratings are available. Invalid numeric values were already replaced with 0 during cleaning.

# 8.Refactoring explanationt

Numeric conversion logic safe_integer() function-la oru thadava write pannirukkom; Boolean conversion safe_boolean()
function-la oru thadava write pannirukkom. Loop moolama ellaa records/fields-kum reuse pannrom. Same logic
repeated-ah ezhudha vendiya avasiyam illa; future change one place-la panna podhum.

# Final validation

In [31]:
print("FINAL VALIDATION")
print("Countries loaded:", len(COUNTRIES))
print("US records:", len(us_records))
print("GB records:", len(gb_records))
print("IN records:", len(in_records))
print("All records:", len(cleaned_records))
print("Categories:", len(category_names))
print("Tuple count:", len(video_tuples))
print("Channel count:", len(channel_counts))
print("Country summaries:", list(country_category_summary.keys()))
print("Classification total:", sum(label_counts.values()))
assert len(cleaned_records) >= 40000
assert len(COUNTRIES) >= 3
assert len(category_names) == 17
assert len(video_tuples) == len(cleaned_records)
assert sum(label_counts.values()) == len(cleaned_records)
print("All final validation checks passed.")

FINAL VALIDATION
Countries loaded: 3
US records: 16080
GB records: 15075
IN records: 15075
All records: 46230
Categories: 17
Tuple count: 46230
Channel count: 438
Country summaries: ['US', 'GB', 'IN']
Classification total: 46230
All final validation checks passed.
